In [1]:
import tensorflow as tf
import tensorflow_recommenders as tfrs

import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

In [2]:
from tqdm.keras import TqdmCallback

c:\Users\bpadmin\anaconda3\envs\atrad_cars_v2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
max_port_size = 50

In [4]:
retriever_location_ = r"D:\dev work\recommender systems\Atrad_CARS\model_weights\2024_07_01_31\retriever_v3_port_v2__fixed_max_port_size_50"
ranking_location_ = r"D:\dev work\recommender systems\Atrad_CARS\model_weights\2024_07_10\tf_listwise_ranking_2024_07_10_16_54"
stock_info_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\stock_data.xlsx"

train_ds_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\retriver_train".format(max_port_size)
test_ds_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\retriver_test".format(max_port_size)
portfolios_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\portfolios".format(max_port_size)


In [5]:
test_ds = tf.data.Dataset.load(test_ds_loc).cache()

train_ds = tf.data.Dataset.load(train_ds_loc).cache()

portfolios = tf.data.Dataset.load(portfolios_loc).cache()

print(f"training dataset size : {len(train_ds)}")
print(f"test dataset size : {len(test_ds)}")
print(f"total size : {len(portfolios)}")

training dataset size : 85393
test dataset size : 14917
total size : 100310


In [19]:
from retrieval_recommender_v3 import Retriever
retriever = Retriever(
    portfolios = portfolios
)
retriever.load_weights(retriever_location_)
retriever.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

from ranker_recommender_v3 import Ranker
ranker = Ranker(
    loss = tf.keras.losses.MeanSquaredError(),
    portfolios = portfolios
)
ranker.load_weights(ranking_location_)
ranker.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))


In [7]:
stock_info = pd.read_excel(stock_info_loc)
stock_info = stock_info.drop(['Unnamed: 0','buisnesssummary'],axis = 1)
stock_info = stock_info.rename(columns = {
    'symbol':'STOCKCODE',
    'name' : 'STOCKNAME',
    'gics_code' : 'GICS'
})
stock_info = stock_info[~stock_info['GICS'].isna()]
print("items data shape : {}".format(stock_info.shape))


unique_items_ = np.unique(np.concatenate(list(train_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator())))
stock_info = stock_info[stock_info['STOCKCODE'].isin([item.decode('utf-8') for item in unique_items_])]


items data shape : (280, 3)


In [8]:
items_ds = tf.data.Dataset.from_tensor_slices(stock_info.to_dict(orient= 'list'))

In [9]:
code2name = dict(zip(stock_info.STOCKCODE, stock_info.STOCKNAME))
code2gics = dict(zip(stock_info.STOCKCODE, stock_info.GICS))

# complete recommender function

In [10]:
items_identiifiers = items_ds.map(lambda x: x["STOCKCODE"])
items_identiifiers = next(iter(items_identiifiers.batch(len(items_identiifiers))))
items_identiifiers.shape

TensorShape([268])

In [11]:
index = tfrs.layers.factorized_top_k.BruteForce(
    query_model = retriever.user_model,
    k = 10)
    
retriever_item_model = retriever.item_model
mapped_items = items_ds.batch(len(items_ds)).map(lambda x : retriever_item_model(x))

mapped_items_tensor = next(iter(mapped_items))
index.index(mapped_items_tensor, items_identiifiers)

In [28]:
def recommend_(test_user):    
    train_items = train_ds.filter(lambda x: tf.reduce_all(tf.math.equal(x["USER_ID"], test_user)))
    seen_items = np.array(list(train_items.map(lambda x : x['STOCKCODE']).as_numpy_iterator())).reshape(1, -1)

    _, recommendations = index.query_with_exclusions(
                            queries = {'USER_ID' : tf.constant([test_user])},
                            exclusions = seen_items
                            )                                                   

    recommendations = [reco.decode('utf-8') for reco in recommendations.numpy().flatten()]

    #code2name and code2gics are two python dictionaries that has STOCKCODE to STOCKNAME and GICS maps.
    names = np.array([code2name[code] for code in recommendations])
    gics = np.array([code2gics[code] for code in recommendations])

    user = {
    'USER_ID' : np.array([test_user]),
    'STOCKCODE' : np.array(recommendations).reshape(-1,10),
    'GICS' : gics.reshape(-1,10),
    'STOCKNAME' : names.reshape(-1,10)
    }

    pred_ratings = ranker(user)
    
    recommendations_w_ratings = pd.DataFrame()
    recommendations_w_ratings['STOCKCODE'] = recommendations
    recommendations_w_ratings['PRED_RATING'] = pred_ratings.numpy().flatten()
    recommendations_w_ratings = recommendations_w_ratings.sort_values( by = ['PRED_RATING'], ascending= False).reset_index(drop = True)
    
    return recommendations_w_ratings

In [44]:
unique_users = np.unique(list(test_ds.map(lambda x: x['CDSACCNO']).as_numpy_iterator()))
unique_user_ids = test_ds.filter(lambda x : x['CDSACCNO'].isin(unique_users)).ma
unique_users

array([[b'COCO', b'EMER', b'SAMP', ..., b'RCL', b'CALT', b'PLC'],
       [b'COCO', b'EMER', b'SAMP', ..., b'RCL', b'CALT', b'PLC'],
       [b'COCO', b'EMER', b'SAMP', ..., b'RCL', b'CALT', b'PLC'],
       ...,
       [b'SEYB', b'HNB', b'AEL', ..., b'ALUM', b'MGT', b'PLC'],
       [b'SEYB', b'HNB', b'AEL', ..., b'ALUM', b'MGT', b'PLC'],
       [b'SEYB', b'HNB', b'AEL', ..., b'ALUM', b'MGT', b'PLC']],
      dtype=object)

In [65]:
cdsaccno_2_userid = dict()
for row in test_ds.as_numpy_iterator():
    cdsaccno_2_userid[row['CDSACCNO']] = row['USER_ID']

In [79]:
import random

random_user_id = cdsaccno_2_userid.get(random.choice(list(cdsaccno_2_userid.keys())))
random_user_id

recommend_(random_user_id)

,STOCKCODE,PRED_RATING
0,HAYL,2.096575
1,NTB,2.013101
2,HNB,1.910473
3,DFCC,1.798719
4,AHPL,1.757041
5,WAPO,1.696092
6,CONN,1.522460
7,TAP,1.514110
8,LALU,1.373652
9,AMF,1.344577


In [16]:
test_timestamp = datetime.timestamp(datetime.now())

total_hit_perc_ = 0
results_ = pd.DataFrame(columns = ['CDSACCNO','hit_perc'])

for user in tqdm(test_df.CDSACCNO.unique()):
    _, hit_perc_ = recommend_(user, test_timestamp)
    results_ = pd.concat(
        [
            results_,
        pd.DataFrame([
            {
                'CDSACCNO':user,
                'hit_perc':hit_perc_
                }
            ])
        ], ignore_index = True)
    total_hit_perc_ += hit_perc_

# (total_hit_perc_/test_df.CDSACCNO.nunique())*100
total_hit_perc_

  0%|          | 0/5906 [00:00<?, ?it/s]C:\Users\naradaw\AppData\Local\Temp\ipykernel_8264\2312754527.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_ = pd.concat(
  0%|          | 6/5906 [00:04<31:27,  3.13it/s]  

  0%|          | 7/5906 [00:04<28:13,  3.48it/s]

100%|██████████| 5906/5906 [05:46<00:00, 17.04it/s]


292.1999999999977

In [17]:
(total_hit_perc_/test_df.CDSACCNO.nunique())*100

4.947511005756819

In [18]:
results_

,CDSACCNO,hit_perc
0,HDF-74565-LI/00,0.1
1,BMS-800262640-VN/00,0.0
2,COM-69742-LC/00,0.1
3,BMS-861802000-VN/00,0.3
4,HDF-743463188-VN/00,0.0
...,...,...
5901,BMS-68660-LI/00,0.0
5902,BMS-683600342-VN/00,0.0
5903,CAS-86452-LI/00,0.1
5904,CMB-5826-LC/00,0.0


In [19]:
results_.hit_perc.value_counts()

hit_perc
0.0    3653
0.1    1724
0.2     426
0.3      79
0.4      17
0.5       4
0.7       1
0.8       1
0.6       1
Name: count, dtype: int64

In [20]:
# results_df_ = pd.DataFrame()

# results_df_['hit_perc'] = results_

# Precision and Recall @k

In [21]:
import array
import collections

from typing import Dict, List, Optional, Text, Tuple

def evaluate(retriever,
             test: tf.data.Dataset,
             train: Optional[tf.data.Dataset] = None,
             timestamp: int = datetime.timestamp(datetime.now()),
             k: int = 10) -> Dict[Text, float]:

  item_ids = stock_info['STOCKCODE'].unique()
  
  item_ids = np.concatenate(
      list(items_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator()))

  item_vocabulary = dict(zip(item_ids.tolist(), range(len(item_ids))))

  train_user_to_items = collections.defaultdict(lambda: array.array("i"))
  test_user_to_items = collections.defaultdict(lambda: array.array("i"))

  if train is not None:
    for row in train.as_numpy_iterator():
      user_id = row["CDSACCNO"]
      item_id = item_vocabulary[row["STOCKCODE"]]
      train_user_to_items[user_id].append(item_id)

  for row in test.as_numpy_iterator():
    user_id = row["CDSACCNO"]
    item_id = item_vocabulary[row["STOCKCODE"]]
    test_user_to_items[user_id].append(item_id)

  item_embeddings = mapped_items_tensor.numpy()

  user_ids = []
  precision_values = []
  recall_values = []
  num_test_items = []
  num_train_items = []

  for user_id, test_items in tqdm(test_user_to_items.items()):
    user_embedding = retriever.user_model(
      (
        tf.constant([user_id]),
        tf.constant([timestamp])
       )
      ).numpy()
    scores = (user_embedding @ item_embeddings.T).flatten()

    test_items = np.frombuffer(test_items, dtype=np.int32)
    
    if train is not None:
      train_items = np.frombuffer(
          train_user_to_items[user_id], dtype=np.int32)
      scores[train_items] = -1e6

    

    top_items = np.argsort(-scores)[:k]
    num_test_items_in_k = sum(x in top_items for x in  test_items)
    precision_values.append(num_test_items_in_k / k)
    recall_values.append(num_test_items_in_k / len(test_items))
    num_test_items.append(len((test_items)))
    num_train_items.append(len(train_user_to_items[user_id]))
    user_ids.append(user_id)

  results_df_ = pd.DataFrame(
    columns = ['CDSACCNO','precision@k', 'recall@k','num_test_items','portfolio_size'],
    data = list(zip(user_ids, precision_values, recall_values, num_test_items, num_train_items))
  )

  return {
      "precision_at_k": np.mean(precision_values),
      "recall_at_k": np.mean(recall_values),
      "results_df_" : results_df_
  }

In [22]:
results = evaluate(
    retriever,
    test_ds,
    train_ds
)

100%|██████████| 5906/5906 [00:41<00:00, 143.02it/s]


In [24]:
results['results_df_']
# results

,CDSACCNO,precision@k,recall@k,num_test_items,portfolio_size
0,b'HDF-74565-LI/00',0.1,0.500000,2,9
1,b'BMS-800262640-VN/00',0.0,0.000000,2,10
2,b'COM-69742-LC/00',0.1,0.100000,10,38
3,b'BMS-861802000-VN/00',0.3,0.214286,14,58
4,b'HDF-743463188-VN/00',0.0,0.000000,2,9
...,...,...,...,...,...
5901,b'BMS-68660-LI/00',0.0,0.000000,3,10
5902,b'BMS-683600342-VN/00',0.0,0.000000,3,10
5903,b'CAS-86452-LI/00',0.1,0.333333,3,10
5904,b'CMB-5826-LC/00',0.0,0.000000,2,8


# Hit Ratio - HOO

In [32]:
def recommend_hoo_(CDSACCNO, timestamp = 1664821800.0):

    seen_items = train_df.loc[train_df.CDSACCNO == CDSACCNO].STOCKCODE.values.reshape(1, -1)

    
    _, recommendations = index.query_with_exclusions(
    queries = (
        tf.constant([CDSACCNO]),
        tf.constant([timestamp])
        ),
    exclusions = seen_items
    )

    recommendations = [reco.decode('utf-8') for reco in recommendations.numpy().flatten()]
    # print(f"Recommendations for user %s: {recommendations}" %(test_user))

    names = np.array([code2name[code] for code in recommendations])
    gics = np.array([code2gics[code] for code in recommendations])

    user = {
    'CDSACCNO' : np.array([CDSACCNO]),
    'STOCKCODE' : np.array(recommendations).reshape(-1,10),
    'GICS' : gics.reshape(-1,10),
    'STOCKNAME' : names.reshape(-1,10)
    }

    pred_ratings = ranker(user)

    recommendations_w_ratings = pd.DataFrame()
    recommendations_w_ratings['STOCKCODE'] = recommendations
    recommendations_w_ratings['PRED_RATING'] = pred_ratings.numpy().flatten()
    recommendations_w_ratings = recommendations_w_ratings.sort_values( by = ['PRED_RATING'], ascending= False).reset_index(drop = True)

    user_test_port_ = test_df.iloc[test_users_.groups[CDSACCNO]].sort_values('RATING', ascending = False)
    recommendations_w_ratings = recommendations_w_ratings.join(user_test_port_.set_index('STOCKCODE'), on = 'STOCKCODE')[['STOCKCODE','PRED_RATING','RATING']]
    
    hit_ = 1 if (recommendations_w_ratings.RATING.isna().all()) == False else 0

    return recommendations_w_ratings, hit_


In [33]:
from tqdm import tqdm
from datetime import datetime

test_timestamp = datetime.timestamp(datetime.now())

hits_ = 0
for user in tqdm(test_df.CDSACCNO.unique()):
    _, hit_ = recommend_hoo_(user, test_timestamp)
    hits_ += hit_

  0%|          | 0/5906 [00:00<?, ?it/s]

100%|██████████| 5906/5906 [07:15<00:00, 13.55it/s]


In [34]:
(hits_/test_df.CDSACCNO.nunique())*100

38.14764646122587

epochs - 11.03962072468676
hoo - 11.395191330849983
